# Week 08 — Home exercise 6: A loader you will call more than once

**Solution proposal.**

Last week's `load_emissions` could not remove the aggregates, because the information was in another
file. This one can — and it is called twice below with different arguments, which is the reason it is
a function with a parameter rather than a block of cells.

In [1]:
import pandas as pd

## The function

In [2]:
def load_panel(emissions_path, info_path, countries_only=True):
    """
    Read the emissions and country files, join them, and return one table.

    Parameters
    ----------
    emissions_path : str
        Path to the emissions CSV file.
    info_path : str
        Path to the country lookup CSV file, holding one row per entity.
    countries_only : bool, optional
        Whether to drop the World Bank aggregate entities such as World and
        Arab World. Default is True, which is what you want for any total,
        average or ranking across entities. Pass False to keep them, for
        example when you want to compare a total you computed against the
        World Bank's own World figure.

    Returns
    -------
    DataFrame
        One row per entity per year, with co2_pc holding emissions in tonnes
        per person, plus the region and income level from the lookup file.
        Rows with no co2_total are dropped.

    Raises
    ------
    ValueError
        If the merge changes the number of rows, which means the lookup file
        is not one row per entity and every per-entity figure downstream
        would be silently wrong.
    """
    emissions = pd.read_csv(emissions_path)
    emissions = emissions.dropna(subset=["co2_total"])

    # co2_total is in millions of tonnes and population is a count of people,
    # so the ratio is scaled by a million to come out in tonnes per person.
    emissions["co2_pc"] = emissions["co2_total"] * 1_000_000 / emissions["population"]

    info = pd.read_csv(info_path)

    # code, not name: two of the names in the lookup file carry a trailing
    # space, and merging on them loses 48 rows without saying anything.
    panel = emissions.merge(
        info[["code", "region", "incomeLevel"]],
        on="code",
        how="left",
        validate="many_to_one",
    )

    if len(panel) != len(emissions):
        raise ValueError(
            f"the merge changed the row count: {len(emissions)} -> {len(panel)}"
        )

    if countries_only:
        panel = panel[panel["region"] != "Aggregates"]

    return panel.reset_index(drop=True)

## Checking both settings

In [3]:
countries = load_panel("../data/co2_emissions.csv", "../data/country_info.csv")
everything = load_panel("../data/co2_emissions.csv", "../data/country_info.csv", countries_only=False)

print("countries only:", countries.shape)
print("everything:    ", everything.shape)
print()
print("entities:", countries["code"].nunique(), "countries out of", everything["code"].nunique())

countries only: (4872, 14)
everything:     (5904, 14)

entities: 203 countries out of 246


4 872 and 5 904, as expected: 203 countries out of 246 entities, with 43 aggregates removed.

The `countries_only=False` setting is not there for symmetry. It is there because the aggregates are
the only way to check your own arithmetic — the `World` row is the World Bank's answer to the question
we have been asking all week.

In [4]:
total_2023 = countries[countries["year"] == 2023]["co2_total"].sum()
world_2023 = everything[
    (everything["country"] == "World") & (everything["year"] == 2023)
]["co2_total"].iloc[0]

print(f"our 203 countries: {total_2023:10,.0f}")
print(f"World Bank World:  {world_2023:10,.0f}")
print(f"difference:        {world_2023 - total_2023:10,.0f}  ({(1 - total_2023 / world_2023) * 100:.1f}%)")

our 203 countries:     37,483
World Bank World:      39,113
difference:             1,630  (4.2%)


Four percent short, and the shortfall is explicable rather than mysterious: 14 entities were dropped
for having no emissions figure at all, and the World Bank's `World` includes territories our 203 do
not.

## Total emissions by region and year

In [5]:
regional = countries.groupby(["region", "year"])["co2_total"].sum().reset_index()

print(regional.shape)
regional.head(3).round(1)

(168, 3)


,region,year,co2_total
0,East Asia & Pacific,2000,6681.3
1,East Asia & Pacific,2001,6907.1
2,East Asia & Pacific,2002,7308.2


`reset_index()` is what turns the two-level index into the two ordinary columns the exercise asked
for — and it is what any plotting or writing to a file will want.

In [6]:
regional[regional["year"] == 2023].sort_values("co2_total", ascending=False).round(0)

,region,year,co2_total
23,East Asia & Pacific,2023,17193.0
47,Europe & Central Asia,2023,6069.0
119,North America,2023,5186.0
95,"Middle East, North Africa, Afghanistan & Pakistan",2023,3242.0
143,South Asia,2023,3174.0
71,Latin America & Caribbean,2023,1748.0
167,Sub-Saharan Africa,2023,871.0


## Emissions per person, by region

The region's **total** emissions over the region's **total** population — which is not the average of
its countries' figures, and is not something any built-in aggregation can compute, because it needs
two columns of the group at once.

So it is a function, and `.apply()` on the groupby hands it each region's rows as a table.

In [7]:
def emissions_per_person(group):
    """Total emissions of a group of rows, per head of its total population."""
    return group["co2_total"].sum() * 1_000_000 / group["population"].sum()


year_2023 = countries[countries["year"] == 2023]

comparison = pd.DataFrame({
    "per_person": year_2023.groupby("region").apply(emissions_per_person),
    "mean_of_countries": year_2023.groupby("region")["co2_pc"].mean(),
})

comparison["difference"] = comparison["per_person"] - comparison["mean_of_countries"]

comparison.round(2).sort_values("difference")

,per_person,mean_of_countries,difference
region,,,
"Middle East, North Africa, Afghanistan & Pakistan",4.09,9.21,-5.12
Latin America & Caribbean,2.66,2.94,-0.28
Sub-Saharan Africa,0.70,0.90,-0.20
South Asia,1.91,1.61,0.29
Europe & Central Asia,6.63,5.81,0.82
East Asia & Pacific,7.28,6.17,1.11
North America,13.76,10.62,3.14


### Where the two disagree, and why

The table is sorted by `difference`, so it splits into two halves.

**Weighting pulls the figure down** for the Middle East and North Africa (−5.12), Latin America
(−0.28) and Sub-Saharan Africa (−0.20). **It pulls the figure up** for South Asia (+0.30), Europe &
Central Asia (+0.82), East Asia & Pacific (+1.11) and North America (+3.14).

**The Middle East and North Africa is the extreme case: 4.09 per person against an unweighted mean of
9.21.** The region contains Qatar (48.6 tonnes per person, 2.7 million people), Bahrain (24.5, 1.6
million) and Kuwait (22.7, 4.9 million) — and also Pakistan, with 248 million people and a very low
figure. The unweighted mean gives Qatar and Pakistan one vote each. The weighted one gives Pakistan
ninety times Qatar's weight, because ninety times as many people live there. **Sub-Saharan Africa
tilts the same way for the same reason**, much more mildly: its high per-person figures belong to
small oil and island states, and its largest populations emit very little.

**North America is the clearest case of the opposite.** Three entities, and the enormous one — the
United States — emits more per head than the small ones, so weighting by population pulls the regional
figure up rather than down.

The rule underneath: **the unweighted mean answers "what does a typical country here look like?" and
the weighted one answers "what does a typical person here emit?"** For anything about climate, the
second is nearly always the question, because the atmosphere counts tonnes and not countries. For
anything about policy or governance, the first often is, because policies are made by countries.

Neither is a default. What is not acceptable is computing one and describing it as the other.

### Things worth noticing

- **The function raises rather than returning something wrong.** A comment saying "the merge should
  not change the row count" is a hope; `raise ValueError` is a check. It costs three lines and it
  converts the worst failure mode of this week — a silently reshaped table — into a stack trace at the
  point of the mistake.
- **`validate="many_to_one"` and the row-count check overlap, and both are worth having.** `validate`
  catches duplicate keys in the lookup; the count check also catches anything else that changes the
  shape, including a future edit to this function.
- **The docstring documents what it raises**, because a caller needs to know that this function can
  fail loudly. A `Raises` section is part of the interface, not decoration.
- `.reset_index(drop=True)` on the way out means the returned table always has a clean 0-to-n index,
  whether or not rows were filtered. A caller should never have to know that aggregates were dropped
  in order to use `.loc` safely.

### What this notebook does NOT do

- **It trusts the lookup file's `region` column completely.** Every country in the output is a country
  because that file says so. If an entity were missing from the lookup, `region` would be `NaN`,
  `NaN != "Aggregates"` is `True`, and it would be quietly kept as a country — the filter would not
  catch it. A stricter version would check `panel["region"].isna().sum() == 0` too.
- **It does not check the columns it depends on exist.** A renamed column in a future download raises a
  `KeyError` from somewhere in the middle of the function rather than a message saying which file was
  the wrong shape.
- **`co2_pc` is computed before the merge and never revisited.** If `population` is missing for a row,
  `co2_pc` comes out as `NaN` and nothing warns you — the column looks complete because the table
  looks complete.
- It has no tests. Nothing checks that `load_panel` on a file with a duplicated code actually raises,
  which is the one behavior the function most needs to be right about.